In [1]:
import platform
import sys

print(platform.system())
print(sys.version)

Linux
3.11.15 (main, Jun  2 2026, 22:26:03) [Clang 22.1.3 ]


In [2]:
import json
from base import Task, Env
from pathlib import Path
from react_agent import ReActAgent
from freeform_plan_and_execute_agent import FreeformPlanAndExecuteAgent
from runner import ReActRunner, FreeformPlanAndExecuteRunner
from langchain_ollama import ChatOllama
from agentic_ai.tools import ALL_TAUBENCH_TOOLS
from agentic_ai.prompts.taubench import WIKI
from agentic_ai.data.taubench_data import load_taubench_data
from agentic_ai.data.taubench_data import TaubenchConnection
from agentic_ai.llm.llm_factory import get_default_llm

In [3]:
file = Path()/"tasks.json"

data = json.loads(file.read_text())

tasks:list[Task] = []
for t in data:
    tasks.append(Task.model_validate(t))
    
print(len(tasks))

115


In [4]:
llm = get_default_llm()
# llm = llm.model_copy(update={
#         "reasoning": True, 
#         "logprobs": True, 
#         "model_kwargs": {"think": "high"}
#     }
# )

print(repr(llm))

ChatOllama(output_version=None, model='gpt-oss:20b', num_ctx=16384, temperature=0.0, keep_alive=-1)


In [5]:
ai_msg = llm.invoke("Hi")
print(ai_msg)

content='Hello! 👋 How can I assist you today?' additional_kwargs={} response_metadata={'model': 'gpt-oss:20b', 'created_at': '2026-07-12T09:58:02.718559922Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1263778413, 'load_duration': 563528523, 'prompt_eval_count': 68, 'prompt_eval_duration': 61062000, 'eval_count': 40, 'eval_duration': 526941000, 'logprobs': None, 'model_name': 'gpt-oss:20b', 'model_provider': 'ollama'} id='lc_run--019f55c3-29e7-7601-be2a-d29b97631ae8-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 68, 'output_tokens': 40, 'total_tokens': 108}


In [6]:
react_agent = ReActAgent(
    model=llm,
    tools=ALL_TAUBENCH_TOOLS,
    wiki=WIKI,
)

# plan_and_execute_agent = FreeformPlanAndExecuteAgent(
#     model=model,
#     tools=ALL_TAUBENCH_TOOLS,
#     wiki=WIKI
# )

In [7]:
from tqdm import tqdm
from pathlib import Path
from contextlib import redirect_stdout

from langsmith import trace, uuid7
from langsmith.client import Client

from dotenv import load_dotenv
assert load_dotenv()

START_INDEX = 87
END_INDEX = None

USER_STRATEGY = "llm_with_prompt"

RUN_IDS = ["5"]

LANGSMITH_PROJECT_NAME = "kltn-agentic-ai"
TAGS = ["taubench"]

client = Client()

for run_id in RUN_IDS:
    print("Running:", run_id)
    LOG_FILE = Path(f".local/act_experimental/run_{run_id}/log/run_{run_id}.log")
    OUTPUT_DIR = Path(f".local/act_experimental/run_{run_id}/output")

    LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    for task_idx, task in enumerate(tqdm(tasks[START_INDEX:END_INDEX], desc="Running", ncols=70), start=START_INDEX):
        
        langsmith_run_id = uuid7()
        
        with trace(
            client=client,
            run_id=langsmith_run_id,
            name=f"task_{task_idx}", 
            project_name=LANGSMITH_PROJECT_NAME,
            tags=TAGS,
            metadata={
                "taubench": {
                    "task_idx": task_idx
                }
            },
        ) as run,\
        TaubenchConnection.session() as session,\
        open(LOG_FILE, "a", encoding="utf-8") as f,\
        redirect_stdout(f):
            env = Env(
                task=tasks[task_idx],
                user_strategy=USER_STRATEGY,
                tools=ALL_TAUBENCH_TOOLS,
                data_load_func=load_taubench_data,
                data=session.data
            )
            
            print(f"\n>>> task_idx: {task_idx} <<<", flush=True)
            
            rs = ReActRunner().run(
                agent=react_agent,
                env=env,
                max_num_steps=20,
                verbose=True
            )
            
            # rs = FreeformPlanAndExecuteRunner().run(
            #     agent=plan_and_execute_agent,
            #     env=env,
            #     max_num_steps=20,
            #     verbose=True
            # )
            
            rs.langsmith = {
                "run_id": run.id,
                "trace_url": run.get_url(),
                "public_url": client.share_run(run_id=run.id)
            }
            
            rs_json = rs.model_dump_json(indent=2, ensure_ascii=True)
            
            ckpt_path = OUTPUT_DIR/f"task_{task_idx}.json"
            _ = ckpt_path.write_text(rs_json)
            
            client.create_feedback(
                    key="reward",
                    score=rs.reward,
                    trace_id=run.id,
                )

            correctness = rs.info["reward_info"]["info"]["correctness"].items()
            for k, v in correctness:
                client.create_feedback(
                    key=k,
                    score=len(v),
                    trace_id=run.id,
                )
            run.add_inputs({
                "messages": rs.messages[:2]
            })
            
            run.add_outputs({
                "messages": rs.messages
            })
            
            run.attachments = {
                "result": (
                    "application/json",
                    rs_json.encode("utf-8")
                )
            }
            
            
            print(repr(rs), flush=True)
            print(f"trace_url:", run.get_url())
            print(f"task {task_idx} saved to:", ckpt_path.absolute().as_uri(), flush=True)
        
        print(f"trace_url:", run.get_url())
        print(f"task {task_idx} saved to:", ckpt_path.absolute().as_uri(), flush=True)

Running: 5


Running:   0%|                                 | 0/28 [00:00<?, ?it/s]

trace_url: https://smith.langchain.com/o/8d3fe009-4227-4f69-9e73-13d1dda2b483/projects/p/8786a668-92b2-4b9f-967a-60e96bf83a5f/r/019f55c3-2f36-7530-8a26-65d8993b2e88?poll=true
task 87 saved to: file:///home/developer/agentic_ai/benchmarks/taubench/baseline/.local/act_experimental/run_5/output/task_87.json


Running:   4%|▉                        | 1/28 [00:49<22:11, 49.30s/it]

trace_url: https://smith.langchain.com/o/8d3fe009-4227-4f69-9e73-13d1dda2b483/projects/p/8786a668-92b2-4b9f-967a-60e96bf83a5f/r/019f55c3-efd0-7b33-aa30-f83b0ace4468?poll=true
task 88 saved to: file:///home/developer/agentic_ai/benchmarks/taubench/baseline/.local/act_experimental/run_5/output/task_88.json


...

Running:  93%|██████████████████████▎ | 26/28 [30:22<02:03, 61.70s/it]

trace_url: https://smith.langchain.com/o/8d3fe009-4227-4f69-9e73-13d1dda2b483/projects/p/8786a668-92b2-4b9f-967a-60e96bf83a5f/r/019f55de-feaf-7840-89a7-e4642046d819?poll=true
task 113 saved to: file:///home/developer/agentic_ai/benchmarks/taubench/baseline/.local/act_experimental/run_5/output/task_113.json


Running:  96%|███████████████████████▏| 27/28 [32:43<01:25, 85.61s/it]

trace_url: https://smith.langchain.com/o/8d3fe009-4227-4f69-9e73-13d1dda2b483/projects/p/8786a668-92b2-4b9f-967a-60e96bf83a5f/r/019f55e1-26fb-7372-b11e-3c2441051325?poll=true
task 114 saved to: file:///home/developer/agentic_ai/benchmarks/taubench/baseline/.local/act_experimental/run_5/output/task_114.json


Running: 100%|████████████████████████| 28/28 [33:22<00:00, 71.52s/it]
